# Download Dataverse File with Guestbook

In [ ]:
from pathlib import Path
import json
import requests
import os

from utils import format_guestbook_template, print_guestbook_form_spec, validate_response, extract_signed_url

## Set Up Intial Variables
**A. Change the following varaibles:**
1. `Dataset_PID` (DOI: see citation)
2. `File_ID` (embeded in file link)

**B. Find your unique API key:**
1. Sign in to [UC Berkeley Library Dataverse](https://datasets.lib.berkeley.edu/)
2. Click your name in the upper right hand corner
3. Select API Token
4. Create an API Token if it is your first time using the API

**C. Set up your environment file:**
1. Copy `.env.example` and rename `.env`
2. Replace value for `API_TOKEN` with your unique Dataverse API Token

In [ ]:
SERVER = 'https://datasets.lib.berkeley.edu'
API_TOKEN = os.environ.get("API_TOKEN")

# Update with the desired dataset DOI
# Must be in the format: doi:10.1234/D3/XXXXXX
DATASET_PID = "doi:10.60503/D3/35DFHU"

# Right click and copy link to desired file for downloading
# Find the fileId embedded in the URL
# e.g. https://datasets.lib.berkeley.edu/file.xhtml?**fileId=33226**&version=7.0
FILE_ID = '33226'

## Get file and Guestbook Metadata

Run the following cells to call the Dataverse API and get metdata about the guestbook. If you do not get a successful response, check your above variables.

In [ ]:
# Get filename and confirm it matches the file you want to download
file_metadata_resp = requests.get(
    f"{SERVER}/api/files/{FILE_ID}/metadata",
    headers={"X-Dataverse-key": API_TOKEN},
    timeout=60,
)
file_metadata_resp.raise_for_status()
filename = file_metadata_resp.json()["label"]

# Display filename
filename

In [ ]:
# Get Guestbook ID applied to the dataset
guestbook_id_resp = requests.get(
    f"{SERVER}/api/datasets/:persistentId/",
    params = {"persistentId": DATASET_PID},
    headers = {"X-dataverse-key": API_TOKEN},
    timeout=60,
)
guestbook_id_resp.raise_for_status()
dataset = guestbook_id_resp.json()["data"]
guestbook_id = dataset.get("guestbookId")

# Get metadata for guestbook
guestbook_formatted_resp = requests.get(
    f"{SERVER}/api/guestbooks/{guestbook_id}/",
    headers = {"X-dataverse-key": API_TOKEN},
    timeout=60
)
guestbook_formatted_resp.raise_for_status()
guestbook_json = guestbook_formatted_resp.json()["data"]

# Display guestbook name
guestbook_json["name"]

## Format Guestbook JSON Response

Steps:
1. Run `json_response_template()` to get the form specification and a properly formatted guestbook response
2. Run `print_guestbook_form_spec()` to see required and valid answers for each question
3. Copy the printed JSON output from `json_response_template()`, paste it into the third cell, and save it as `completed_guestbook_response`
4. Validate your completed response against the `form_spec` using `validate_response()`

Functions in this section are imported from `utils.py`.

In [ ]:
# Run cell to print template JSON response
form_spec, json_response_template = format_guestbook_template(guestbook_json)

print(json.dumps(json_response_template, indent=2))

In [ ]:
# Run cell to see custom questions (if any)
print_guestbook_form_spec(form_spec)

In [ ]:
# Copy template JSON response into cell and fill out with valid answers
completed_guestbook_response = {}

In [ ]:
# Run cell after completing template to ensure it is valid
validate_response(form_spec, completed_guestbook_response)

## Post Guestbook to API and Download file

Run final cell to download file

In [ ]:
# Set up variables for signed URL request
endpoint = f"{SERVER}/api/access/datafile/{FILE_ID}"
params = {
    "signed": "true",
}
payload = {
    "guestbookResponse": completed_guestbook_response
}

# Request signed URL for file download
signed_url_resp = requests.post(
    endpoint, 
    params=params, 
    json=payload,
    headers={"X-Dataverse-key": API_TOKEN, "Accept": "application/json"},
    timeout=120)
print("Signed URL Status:", signed_url_resp.status_code)
signed_url_resp.raise_for_status()
signed_url_json = signed_url_resp.json()
signed_download_url = extract_signed_url(signed_url_json)

# Make directory for downloaded file if it doesn't exist
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Download file using signed URL, save to data directory, and display progress
with requests.get(signed_download_url, stream=True, timeout=(30, 3600)) as resp:
    print("Download status:", resp.status_code)
    resp.raise_for_status()

    output_path = data_dir / filename

    total = 0
    with output_path.open("wb") as f:
        for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                f.write(chunk)
                total += len(chunk)
                print(f"\rDownloaded {total / (1024 * 1024):.1f} MiB", 
                      end="", flush=True
                     )

print(f"\nSaved to {output_path}")